In [0]:
%run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_3", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_3", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
cd $HOME;
if [ ! -d Isaac-GR00T ]; then
  git clone --recurse-submodules https://github.com/NVIDIA/Isaac-GR00T;
else
  cd Isaac-GR00T && git fetch;
fi;
ls -la $HOME/Isaac-GR00T;
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
cd $HOME/Isaac-GR00T;
# The repo ships a local aarch64 flash-attn wheel that breaks uv run on x86_64.
# Remove it so uv resolves flash-attn from PyPI/GitHub instead.
sed -i '/flash.attn.*aarch64/d' pyproject.toml;
echo "== flash-attn lines remaining in pyproject.toml ==";
grep -i 'flash' pyproject.toml || echo "(none - correctly removed)";
# Regenerate the lock file to pick up the change
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
uv lock;
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
cd $HOME/Isaac-GR00T;
echo == Torch CUDA check ==;
.venv/bin/python -c 'import torch; print("torch:", torch.__version__); print("cuda available:", torch.cuda.is_available())';
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP  << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
CUDA_PATH=$(ls -d /usr/local/cuda-* 2>/dev/null | grep -E '/usr/local/cuda-[0-9]' | sort -V | tail -1);
if [ -z "$CUDA_PATH" ] && [ -e /usr/local/cuda ]; then CUDA_PATH=$(readlink -f /usr/local/cuda); fi;
export CUDA_HOME=${CUDA_PATH:-/usr/local/cuda};
export PATH=$CUDA_HOME/bin:$PATH;
export LD_LIBRARY_PATH=$CUDA_HOME/lib64:$LD_LIBRARY_PATH;
export TORCH_CUDA_ARCH_LIST="8.0;8.6;8.9;9.0";
export MAX_JOBS=$(nproc);
cd $HOME/Isaac-GR00T;
echo "== Using CUDA_HOME: $CUDA_HOME ==";
uv pip install ninja packaging;
echo == flash-attn import check ==;
.venv/bin/python -c 'import flash_attn; print("flash_attn OK:", flash_attn.__version__)';
echo == Setup complete ==;
EOF

In [ ]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin";
cd $HOME/Isaac-GR00T;
.venv/bin/huggingface-cli download nvidia/GR00T-N1.7-3B;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
export PATH="$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
cd $HOME/Isaac-GR00T;
uv run hf download nvidia/GR00T-N1.7-3B;
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_3 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
cd $HOME/Isaac-GR00T;
mkdir -p finetuned_models;
ls -la finetuned_models;
EOF